# Diabetic Retinopathy Detection System (Corrected Version)
This notebook extracts the Kaggle dataset, preprocesses retinal images using CLAHE on the green channel, uses a DDPM diffusion model to balance minority classes, trains a ResNet18 classifier, and launches a Gradio app for user testing.


### 1. Mount Google Drive


In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Failed to mount Google Drive:", e)


### 2. Configure Folder Paths


In [ ]:
BASE_PATH = "/content/drive/MyDrive/DR_Project"
DATASET_ZIP = "/content/diabetic-retinopathy-dataset.zip"
EXTRACT_PATH = "/content/dataset"
BALANCED_PATH = BASE_PATH + "/balanced_dataset"
MODEL_PATH = BASE_PATH + "/models"

os.makedirs(BALANCED_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
print("Directories initialized:")
print(" - Balanced Dataset Path:", BALANCED_PATH)
print(" - Checkpoints Path:", MODEL_PATH)


### 3. Configure Kaggle Credentials Securely
> **Note:** To secure your API key, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Google Colab secrets manager (key icon in the left toolbar) and turn on access. The code below retrieves them securely.


In [ ]:
import json
from google.colab import userdata

try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
    
    if not kaggle_username or not kaggle_key:
        raise ValueError("Credentials not found in secrets.")
    
    kaggle_json = {
        "username": kaggle_username,
        "key": kaggle_key
    }
    print("Loaded Kaggle credentials securely from Colab Secrets.")
except Exception as e:
    print("Warning: Colab Secrets not found or configured. Falling back to pre-configured developer credentials.")
    # Developer credentials fallback
    kaggle_json = {
        "username": "keer26",
        "key": "KGAT_78563d4fdf688ee7fa46af86b3809c18"
    }

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_json, f)
os.chmod("/root/.kaggle/kaggle.json", 600)
print("Kaggle configuration file written.")


### 4. Download Dataset from Kaggle


In [ ]:
!pip install -q kaggle
!kaggle datasets download -d sachinkumar413/diabetic-retinopathy-dataset -p /content


### 5. Extract Dataset
*This step has been reordered to run immediately after downloading, preventing FileNotFoundError errors during image preprocessing visualization.*


In [ ]:
import zipfile

if not os.path.exists(EXTRACT_PATH):
    print("Extracting dataset zip...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extraction completed successfully.")
else:
    print("Dataset already extracted at:", EXTRACT_PATH)


### 6. Image Preprocessing and Visualization
Preprocesses raw images by converting to RGB, extracting the green channel (where DR features like microaneurysms and hemorrhages are most visible), applying CLAHE for contrast enhancement, and a Gaussian blur to reduce high-frequency noise.


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    resized = cv2.resize(img, (224, 224))
    green = resized[:, :, 1]

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(green)

    blurred = cv2.GaussianBlur(clahe_img, (5,5), 0)

    return img, blurred

dataset_path = "/content/dataset"
classes = [
    f for f in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, f))
]

print("Classes:", classes)

plt.figure(figsize=(10, 15))
for i, cls in enumerate(classes):
    class_dir = os.path.join(dataset_path, cls)
    img_name = random.choice(os.listdir(class_dir))
    img_path = os.path.join(class_dir, img_name)

    original, processed = preprocess_image(img_path)

    # Original
    plt.subplot(len(classes), 2, 2*i + 1)
    plt.imshow(original)
    plt.title(f"{cls} - Original")
    plt.axis("off")

    # Processed
    plt.subplot(len(classes), 2, 2*i + 2)
    plt.imshow(processed, cmap='gray')
    plt.title(f"{cls} - Preprocessed")
    plt.axis("off")

plt.tight_layout()
plt.show()


### 7. Initialize Balanced Dataset Directory
Copies all original downloaded images into the balanced dataset directory before DDPM generation adds synthetic minority images.


In [ ]:
import shutil

for cls in os.listdir(EXTRACT_PATH):
    src = os.path.join(EXTRACT_PATH, cls)
    dst = os.path.join(BALANCED_PATH, cls)
    os.makedirs(dst, exist_ok=True)

    for img in os.listdir(src):
        src_file = os.path.join(src, img)
        dst_file = os.path.join(dst, img)
        if not os.path.exists(dst_file):
            shutil.copy(src_file, dst)

print("Original images copied to:", BALANCED_PATH)


### 8. Custom PyTorch Retinal Preprocessing Transform
Defines a custom preprocessing class to integrate your green-channel + CLAHE + blur preprocessing workflow directly into the PyTorch `Dataloader` pipeline. This ensures your neural network trains and runs inference on preprocessed images.


In [ ]:
import numpy as np
from PIL import Image

class RetinalPreprocessing(object):
    def __call__(self, img):
        img_np = np.array(img)
        # Ensure image has 3 color channels
        if len(img_np.shape) == 2:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
        elif img_np.shape[2] == 4:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGBA2RGB)
            
        resized = cv2.resize(img_np, (224, 224))
        green = resized[:, :, 1]
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        clahe_img = clahe.apply(green)
        
        blurred = cv2.GaussianBlur(clahe_img, (5,5), 0)
        
        # Merge back to a 3-channel grayscale for ResNet architecture compatibility
        processed_3ch = cv2.merge([blurred, blurred, blurred])
        return Image.fromarray(processed_3ch)

print("Custom RetinalPreprocessing transform is ready.")


### 9. Fast DDPM (Diffusion) Setup & Configuration


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from diffusers import UNet2DModel, DDPMScheduler, DDPMPipeline
from torch import nn
from tqdm import tqdm
import glob

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DDPM running on device:", DEVICE)

IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 20
TIMESTEPS = 200

# Normalization transform for DDPM generator
ddpm_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


### 10. Initialize DDPM Model and Data Loader
Configures the UNet2DModel architecture with Dual Attention layers.


In [ ]:
def get_model():
    model = UNet2DModel(
        sample_size=IMG_SIZE,
        in_channels=3,
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(32, 64, 128),
        down_block_types=(
            "DownBlock2D",
            "AttnDownBlock2D",
            "DownBlock2D"
        ),
        up_block_types=(
            "UpBlock2D",
            "AttnUpBlock2D",
            "UpBlock2D"
        ),
    ).to(DEVICE)
    return model

def get_class_loader(class_name):
    dataset = datasets.ImageFolder(EXTRACT_PATH, transform=ddpm_transform)
    # Get exact folder mappings to avoid key mismatches
    class_idx = dataset.class_to_idx[class_name]

    indices = [i for i, (_, label) in enumerate(dataset) if label == class_idx]
    subset = Subset(dataset, indices)

    loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)
    return loader

def load_latest_checkpoint(model, class_name):
    # Sanitize class name for file path safety
    safe_cls = class_name.replace(" ", "_")
    files = glob.glob(f"{MODEL_PATH}/ddpm_{safe_cls}_epoch_*.pth")
    if len(files) == 0:
        return 0

    latest = sorted(files)[-1]
    epoch_num = int(latest.split("_")[-1].split(".")[0])

    model.load_state_dict(torch.load(latest, map_location=DEVICE))
    print(f"Resumed {class_name} from epoch {epoch_num}")

    return epoch_num + 1


### 11. Train DDPM Generator


In [ ]:
def train_ddpm(class_name):
    print(f"\nTraining DDPM generator for {class_name}...")

    loader = get_class_loader(class_name)
    model = get_model()
    scheduler = DDPMScheduler(num_train_timesteps=TIMESTEPS)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.MSELoss()

    start_epoch = load_latest_checkpoint(model, class_name)
    model.train()

    for epoch in range(start_epoch, EPOCHS):
        pbar = tqdm(loader)

        for images, _ in pbar:
            images = images.to(DEVICE)

            noise = torch.randn_like(images)
            timesteps = torch.randint(0, TIMESTEPS, (images.size(0),), device=DEVICE)

            noisy_images = scheduler.add_noise(images, noise, timesteps)
            noise_pred = model(noisy_images, timesteps).sample

            loss = loss_fn(noise_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_description(f"{class_name} Epoch {epoch} Loss {loss.item():.4f}")

        safe_cls = class_name.replace(" ", "_")
        save_file = f"{MODEL_PATH}/ddpm_{safe_cls}_epoch_{epoch}.pth"
        torch.save(model.state_dict(), save_file)
        print("Saved model checkpoint:", save_file)

    return model, scheduler


### 12. Synthetic Image Generation function


In [ ]:
def generate_images(model, scheduler, class_name, num_images):
    if num_images <= 0:
        print(f"Skipping generation for {class_name} (already balanced)")
        return
        
    print(f"Generating {num_images} synthetic images for {class_name} to balance the class...")
    pipeline = DDPMPipeline(unet=model, scheduler=scheduler).to(DEVICE)

    save_path = os.path.join(BALANCED_PATH, class_name)
    os.makedirs(save_path, exist_ok=True)

    existing = len(os.listdir(save_path))

    for i in tqdm(range(num_images), desc=f"Generating {class_name}"):
        image = pipeline(num_inference_steps=100).images[0]
        image.save(f"{save_path}/synthetic_{existing+i}.png")

    print(f"Generation completed for {class_name}.")


### 13. Dynamically Compute Balancing Plan
*Calculates the exact number of synthetic images needed for each class to reach 1000 images, correcting the hardcoded plan imbalance.*


In [ ]:
target_count = 1000
generate_plan = {}

# Check existing folder paths in BALANCED_PATH
for cls in os.listdir(BALANCED_PATH):
    cls_path = os.path.join(BALANCED_PATH, cls)
    if os.path.isdir(cls_path):
        current_count = len(os.listdir(cls_path))
        needed = target_count - current_count
        if needed > 0:
            generate_plan[cls] = needed

print("Dynamic generation targets:")
for cls, count in generate_plan.items():
    print(f" - {cls}: generate {count} images to reach 1000")


### 14. Train and Generate Synthetic Images for Imbalanced Classes
Trains a DDPM generator on each minority class in a loop and generates the exact count needed to balance them.


In [ ]:
for cls, count in generate_plan.items():
    # Train the model for this class
    model, scheduler = train_ddpm(cls)
    # Generate the missing synthetic images
    generate_images(model, scheduler, cls, count)


### 15. Verify Balanced Dataset Sizes


In [ ]:
print("Current size of classes in balanced_dataset:")
for cls in os.listdir(BALANCED_PATH):
    cls_path = os.path.join(BALANCED_PATH, cls)
    if os.path.isdir(cls_path):
        print(f" - {cls}: {len(os.listdir(cls_path))} images")


### 16. Prepare Classifier Setup


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split
from collections import Counter
from tqdm import tqdm

DATA_PATH = "/content/drive/MyDrive/DR_Project/balanced_dataset"
MODEL_PATH = "/content/drive/MyDrive/DR_Project/dr_classifier_bias_reduced.pth"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Classifier running on device:", device)


### 17. Configure Classifier Data Pipelines
Defines training transform with retinal preprocessing + augmentations, and validation transform with retinal preprocessing + normalization only (no augmentations). Uses standard ImageNet statistics.


In [ ]:
train_transform = transforms.Compose([
    RetinalPreprocessing(), # Apply CLAHE, green channel, and Gaussian blur
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1), # Applied gently on preprocessed channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet stats
])

val_transform = transforms.Compose([
    RetinalPreprocessing(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


### 18. Train/Validation Split and Resampling
Splits the dataset into 80% training and 20% validation. Resolves the double-compensation class bias problem by using `WeightedRandomSampler` **only** to balance batches during training. Validation data uses standard un-augmented data loader.


In [ ]:
# Load overall dataset without transform first
full_dataset = datasets.ImageFolder(DATA_PATH)
classes = full_dataset.classes
num_classes = len(classes)
print("Classes:", classes)

# Split 80/20 train/validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_sub, val_sub = random_split(full_dataset, [train_size, val_size])

# Custom wrapper to apply separate train/val transforms to splits
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_sub, transform=train_transform)
val_dataset = TransformSubset(val_sub, transform=val_transform)

# Calculate sampler weights for the training subset
train_indices = train_sub.indices
train_labels = [full_dataset.samples[i][1] for i in train_indices]
class_counts = Counter(train_labels)
total_train = len(train_labels)

class_weights = {k: total_train / v for k, v in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Define Loaders
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Prepared data split:")
print(f" - Training samples: {len(train_dataset)}")
print(f" - Validation samples: {len(val_dataset)}")


### 19. Load Pretrained ResNet18
Uses modern PyTorch `weights` parameter to load ResNet18 pretrained on ImageNet.


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Standard CrossEntropyLoss (we do NOT use class weight parameters here
# because WeightedRandomSampler has already balanced class distributions in batches)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


### 20. Train the Classifier
Trains ResNet18 for 12 epochs, validating metrics on the clean validation set at the end of each epoch.


In [ ]:
EPOCHS = 12

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, labels_batch in pbar:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_description(f"Epoch {epoch+1} Loss {loss.item():.4f}")

    # Evaluate validation accuracy
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)
            
            val_outputs = model(val_images)
            val_preds = torch.argmax(val_outputs, dim=1)
            
            val_correct += (val_preds == val_labels).sum().item()
            val_total += val_labels.size(0)

    val_accuracy = (val_correct / val_total) * 100
    print(f"Epoch {epoch+1} Avg Loss: {running_loss/len(train_loader):.4f} | Validation Accuracy: {val_accuracy:.2f}%")


### 21. Save the Trained Model


In [ ]:
torch.save(model.state_dict(), MODEL_PATH)
print("Bias-reduced ResNet18 model successfully saved at:", MODEL_PATH)


### 22. Evaluate Model Performance on the Clean Validation Set
Outputs precision, recall, f1-score, and accuracy on the validation set, ensuring evaluation metrics are statistically valid.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"\nOverall Validation Accuracy: {acc*100:.2f} %")
print("\nClassification Report (Validation Split):")
print(classification_report(all_labels, all_preds, target_names=classes))


### 23. Recommended Precautions Helper


In [ ]:
def get_precautions(stage):
    precautions = {
        "Healthy": [
            "Maintain normal blood sugar levels",
            "Have annual eye checkups",
            "Follow healthy diet and exercise regularly",
            "Avoid smoking and alcohol"
        ],
        "Mild DR": [
            "Monitor blood sugar strictly",
            "Visit eye specialist every 6-12 months",
            "Control blood pressure and cholesterol",
            "Increase physical activity"
        ],
        "Moderate DR": [
            "Frequent retinal examinations (every 3-6 months)",
            "Strict diabetes control required",
            "Follow doctor-prescribed medication",
            "Avoid high sugar and processed foods"
        ],
        "Severe DR": [
            "Immediate consultation with ophthalmologist",
            "Strict glucose and BP control",
            "Possible laser treatment evaluation",
            "Do not ignore vision changes"
        ],
        "Proliferate DR": [
            "Urgent specialist consultation required",
            "Possible laser or injection treatment",
            "Monitor vision regularly",
            "Strict diabetes management is critical"
        ]
    }
    return precautions.get(stage, ["Consult doctor"])


### 24. Manual Upload and Model Inference Test


In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
if len(uploaded) > 0:
    img_name = list(uploaded.keys())[0]

    image = Image.open(img_name).convert('RGB')
    image_tensor = val_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()

    stage = classes[pred_idx]
    print("\nPredicted DR Stage:", stage)

    print("\nClass Probabilities:")
    for i, cls in enumerate(classes):
        print(f" - {cls}: {probs[0][i].item():.4f}")

    print("\nRecommended Precautions:")
    for i, tip in enumerate(get_precautions(stage), 1):
        print(f"{i}. {tip}")


### 25. Launch Gradio Interactive Application
*Runs Gradio in a self-contained environment to avoid out-of-order execution NameError bugs.*


In [ ]:
!pip install -q gradio

import gradio as gr
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Wrapper transform class for Gradio prediction matching training pipeline
class RetinalPreprocessing(object):
    def __call__(self, img):
        img_np = np.array(img)
        if len(img_np.shape) == 2:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
        elif img_np.shape[2] == 4:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGBA2RGB)
            
        resized = cv2.resize(img_np, (224, 224))
        green = resized[:, :, 1]
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        clahe_img = clahe.apply(green)
        
        blurred = cv2.GaussianBlur(clahe_img, (5,5), 0)
        processed_3ch = cv2.merge([blurred, blurred, blurred])
        return Image.fromarray(processed_3ch)

gradio_transform = transforms.Compose([
    RetinalPreprocessing(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load model structure
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 5) # 5 classes
model = model.to(device)

MODEL_PATH = "/content/drive/MyDrive/DR_Project/dr_classifier_bias_reduced.pth"
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Model Loaded Successfully ✅")

classes = ['Healthy', 'Mild DR', 'Moderate DR', 'Proliferate DR', 'Severe DR']

precautions_map = {
    "Healthy": "Maintain healthy lifestyle and regular checkups.",
    "Mild DR": "Control blood sugar and yearly screening.",
    "Moderate DR": "Consult doctor regularly.",
    "Proliferate DR": "Urgent treatment needed.",
    "Severe DR": "Immediate medical attention required."
}

def predict(image):
    try:
        image = image.convert("RGB")
        x = gradio_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(x)
            pred = torch.argmax(output, 1).item()

        stage = classes[pred]
        precaution = precautions_map[stage]

        return stage, precaution

    except Exception as e:
        return f"Error: {str(e)}", "Check model / input"

interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Retinal Image"),
    outputs=[
        gr.Text(label="Predicted DR Stage"),
        gr.Text(label="Precautions")
    ],
    title="Diabetic Retinopathy Detection System",
    description="Upload retinal image to detect DR stage"
)

interface.launch(share=True)
